Assignment:
1. Load daily prices for [AAPL, MSFT, NVDA, SPX]
2. Form correlations of 1W returns
3. Calculate isopremium (equal premium) strikes for given vols (ATM, skew, kurt)
4. Calculate percentage of realized 1M returns above isopremium line

In [82]:
import math
import numpy as np
import pandas as pd
from os.path import join
import plotly.express as px

data_dir = join("../../", "market_data", "daily")

In [83]:
assets = ["AAPL", "MSFT", "NVDA", "SPX"]

### Load daily prices

In [84]:
series = {}
for asset in assets:
    file_path = join(data_dir, f"{asset}.csv")

    df = pd.read_csv(file_path, parse_dates=["Date"])
    df = df.sort_values("Date")
    series[asset] = df.set_index("Date")["Close"].astype(float)

prices = pd.concat(series, axis=1).dropna(axis=0)
prices = prices[assets]

print("Daily prices shape:", prices.shape)
print(prices.tail())


Daily prices shape: (291, 4)
                  AAPL        MSFT        NVDA      SPX
Date                                                   
2026-02-25  274.230011  400.600006  195.549408  6946.13
2026-02-26  272.950012  401.720001  184.879990  6908.86
2026-02-27  264.179993  392.739990  177.180420  6878.88
2026-03-02  264.720001  398.549988  182.470123  6881.62
2026-03-03  263.750000  403.929993  180.040253  6816.63


/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_12720/4133518819.py:9: Pandas4Warning:

Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.



### Visualize 1-week correlations

In [85]:
# 1-week returns ~= 5 trading days
returns_1w = prices.pct_change(5).dropna()
corr_1w = returns_1w.corr()

fig = px.imshow(
    corr_1w,
    text_auto=".2f",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title="Correlation Matrix of 1-Week Returns",
)
fig.update_layout(coloraxis_colorbar_title="Corr")
fig.show()

corr_1w

,AAPL,MSFT,NVDA,SPX
AAPL,1.000000,0.145083,0.248304,0.642447
MSFT,0.145083,1.000000,0.597384,0.620207
NVDA,0.248304,0.597384,1.000000,0.696144
SPX,0.642447,0.620207,0.696144,1.000000


### Visualize scatterplot of 1W returns for 2 most correlated assets

In [ ]:

# Scatter of 1W returns: SPX vs NVDA + linear regression line
plot_df = returns_1w[["SPX", "NVDA"]].dropna().copy()

# OLS slope/intercept via polyfit
iso_slope, intercept = np.polyfit(plot_df["SPX"], plot_df["NVDA"], 1)
plot_df["NVDA_fit"] = iso_slope * plot_df["SPX"] + intercept

fig = px.scatter(
    plot_df,
    x="SPX",
    y="NVDA",
    title="1W Returns: SPX vs NVDA",
    labels={"SPX": "SPX 1W Return", "NVDA": "NVDA 1W Return"},
)

fit_line = plot_df.sort_values("SPX")
fig.add_scatter(
    x=fit_line["SPX"],
    y=fit_line["NVDA_fit"],
    mode="lines",
    name=f"Linear fit (beta={iso_slope:.2f})",
    line=dict(width=3),
)

fig.show()

In [87]:
# set vol levels and smile for SPX and NVDA
vols = dict()
vols["SPX"] = {"atm": 0.15, "skew": -0.03, "kurt": 0.01}
vols["NVDA"] = {"atm": 0.32, "skew": -0.05, "kurt": 0.02}

r = 0.0
T = 7 / 365  # 1W expiry in years

In [88]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_call_price(S: float, K: float, T: float, r: float, sigma: float) -> float:
    if sigma <= 0 or T <= 0:
        return max(S - K, 0.0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)


def solve_local_vol(S: float, K: float, T: float, r: float, atm: float, skew: float, kurt: float,
                    max_iter: int = 100, tol: float = 1e-10) -> tuple[float, float]:
    # Fixed-point solve because sigma depends on d1 and d1 depends on sigma
    sigma = max(atm, 1e-6)
    for _ in range(max_iter):
        d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
        new_sigma = atm + skew * d1 + kurt * d1 * d1
        new_sigma = max(new_sigma, 1e-6)
        if abs(new_sigma - sigma) < tol:
            sigma = new_sigma
            break
        sigma = new_sigma

    d1_final = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    return sigma, d1_final

In [89]:
spots = prices.iloc[-1][["SPX", "NVDA"]].to_dict()
rows = []

# Strikes as % above spot: 0% .. +15%
strike_pct_grid = np.arange(0.00, 0.16, 0.01)

for asset in ["SPX", "NVDA"]:
    S = float(spots[asset])
    atm = vols[asset]["atm"]
    skew = vols[asset]["skew"]
    kurt = vols[asset]["kurt"]

    for pct_above_spot in strike_pct_grid:
        K = S * (1.0 + pct_above_spot)
        sigma, d1 = solve_local_vol(S, K, T, r, atm, skew, kurt)
        call = bs_call_price(S, K, T, r, sigma)
        rows.append({
            "asset": asset,
            "spot": S,
            "strike": K,
            "strike_pct_above_spot": pct_above_spot,
            "d1": d1,
            "vol": sigma,
            "call_price": call,
            "call_price_pct_spot": call / S,
        })

options_df = pd.DataFrame(rows)

In [90]:
# Isopremium mapping:
# for each SPX strike %, find NVDA strike % with same call price as % of spot.

def call_price_pct_for_asset(asset: str, strike_pct_above_spot: float) -> float:
    S = float(prices.iloc[-1][asset])
    K = S * (1.0 + strike_pct_above_spot)
    atm = vols[asset]["atm"]
    skew = vols[asset]["skew"]
    kurt = vols[asset]["kurt"]

    sigma, _ = solve_local_vol(S, K, T, r, atm, skew, kurt)
    call = bs_call_price(S, K, T, r, sigma)
    return call / S


def solve_nvda_strike_pct_for_target_call_pct(target_call_pct: float,
                                              low: float = 0.0,
                                              high: float = 0.20,
                                              tol: float = 1e-10,
                                              max_iter: int = 100) -> float:
    # solve for strike % above spot with simple dichotomy method
    f_low = call_price_pct_for_asset("NVDA", low) - target_call_pct
    f_high = call_price_pct_for_asset("NVDA", high) - target_call_pct

    # If no sign change, no solution in [low, high]
    if f_low * f_high > 0:
        return np.nan

    lo, hi = low, high
    flo, fhi = f_low, f_high

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = call_price_pct_for_asset("NVDA", mid) - target_call_pct

        if abs(fmid) < tol or (hi - lo) < tol:
            return mid

        if flo * fmid <= 0:
            hi, fhi = mid, fmid
        else:
            lo, flo = mid, fmid

    return 0.5 * (lo + hi)


spx_strike_pcts = np.sort(
    options_df.loc[options_df["asset"] == "SPX", "strike_pct_above_spot"].unique()
)

iso_rows = []
for spx_pct in spx_strike_pcts:
    spx_call_pct = call_price_pct_for_asset("SPX", float(spx_pct))
    nvda_pct = solve_nvda_strike_pct_for_target_call_pct(spx_call_pct)

    nvda_call_pct = np.nan if np.isnan(nvda_pct) else call_price_pct_for_asset("NVDA", float(nvda_pct))

    iso_rows.append({
        "spx_strike_pct": float(spx_pct),
        "spx_call_price_pct_spot": spx_call_pct,
        "nvda_strike_pct_iso": nvda_pct,
        "nvda_call_price_pct_spot": nvda_call_pct,
    })

isopremium_df = pd.DataFrame(iso_rows)
isopremium_df

,spx_strike_pct,spx_call_price_pct_spot,nvda_strike_pct_iso,nvda_call_price_pct_spot
0,0.00,0.008270,0.030722,0.008270
1,0.01,0.004994,0.051606,0.004994
2,0.02,0.003042,0.072860,0.003042
3,0.03,0.001880,0.094323,0.001880
4,0.04,0.001181,0.115947,0.001181
5,0.05,0.000753,0.137723,0.000753
6,0.06,0.000488,0.159651,0.000488
7,0.07,0.000320,0.181737,0.000320
8,0.08,0.000213,NaN,NaN
9,0.09,0.000143,NaN,NaN


In [91]:
# plot isopremium line on the previous figure
fig.add_scatter(
    x=isopremium_df["spx_strike_pct"],
    y=isopremium_df["nvda_strike_pct_iso"],
    mode="lines",
    name="Isopremium line",
    line=dict(width=3),
)
fig.show()

In [97]:
# calculate the proportion of realized 1W NVDA returns above isopremium line
iso_slope = isopremium_df["nvda_strike_pct_iso"].diff(4).dropna().iloc[0] / isopremium_df["spx_strike_pct"].diff(4).dropna().iloc[0]

intercept = isopremium_df["nvda_strike_pct_iso"].iloc[0]

# equation of the isopremium line:
# nvda_pct = iso_slope * spx_pct + intercept
def isopremium_line(spx_pct: float) -> float:
    return iso_slope * spx_pct + intercept


In [100]:
# calculate the proportion of realized 1W NVDA returns above isopremium line
realized_1w_returns = prices.pct_change(5)[["SPX", "NVDA"]]
# keep positive returns only
realized_1w_returns = realized_1w_returns[realized_1w_returns["SPX"] > 0]
realized_1w_returns["isopremium"] = isopremium_line(realized_1w_returns["SPX"])

realized_1w_returns["above_isopremium"] = realized_1w_returns["NVDA"] > realized_1w_returns["isopremium"]

above_isopremium = realized_1w_returns["above_isopremium"].mean()

print(f"Proportion of realized 1W NVDA returns above isopremium line: {above_isopremium:.2%}")

Proportion of realized 1W NVDA returns above isopremium line: 21.23%
